## Preprocessing

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

In [9]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df = pd.read_csv(SPLITS_DIR / "val.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)

In [10]:
def load_and_preprocess(image_path, mask_path):
    image_path = PROJECT_ROOT / image_path
    mask_path = PROJECT_ROOT / mask_path

    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")

    image = np.array(image, dtype=np.float32)
    mask = np.array(mask, dtype=np.uint8)

    # Scale RGB values from [0, 255] to [0, 1].
    image = image / 255.0

    # Normalize using ImageNet statistics for the pretrained ResNet18 encoder.
    image = (image - IMAGENET_MEAN) / IMAGENET_STD

    # Convert mask values from {0, 255} to {0, 1}.
    mask = (mask > 0).astype(np.float32)

    # Convert NumPy arrays to PyTorch tensors.
    # H x W x C -> C x H x W
    image = torch.from_numpy(image).permute(2, 0, 1)

    # H x W -> 1 x H x W
    mask = torch.from_numpy(mask).unsqueeze(0)

    return image, mask

In [11]:
sample = train_df.iloc[0]

image, mask = load_and_preprocess(
    sample["image_path"],
    sample["mask_path"]
)

print("Image shape:", image.shape)
print("Image dtype:", image.dtype)
print("Image min:", image.min().item())
print("Image max:", image.max().item())

print()

print("Mask shape:", mask.shape)
print("Mask dtype:", mask.dtype)
print("Mask values:", torch.unique(mask))

Image shape: torch.Size([3, 256, 256])
Image dtype: torch.float32
Image min: -2.1179039478302
Image max: 2.6051416397094727

Mask shape: torch.Size([1, 256, 256])
Mask dtype: torch.float32
Mask values: tensor([0., 1.])


## Preprocessing summary

Each 256 × 256 RGB image patch is converted to `float32` and first scaled
from the original `[0, 255]` range to `[0, 1]`.

Because the baseline uses a ResNet18 encoder pretrained on ImageNet,
the RGB channels are then normalized using the ImageNet mean and standard
deviation.

The image layout is converted from `H × W × C` to the PyTorch format
`C × H × W`.

Each segmentation mask is converted to a single-channel binary mask:

- `0` — background
- `1` — building

and reshaped to `1 × H × W`.

No resizing is required because all processed patches already have size
256 × 256.
